In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

In [ ]:
# Load the incident-level dataset with weather already joined to each incident
df = pd.read_csv('reduced_incident_weather_merged.csv', parse_dates=['date'])

# Define the weather columns we want to keep after aggregation
weather_cols = ['Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)', 'Total Rain (mm)',
                'Total Snow (cm)', 'Total Precip (mm)', 'Snow on Grnd (cm)',
                'Dir of Max Gust (10s deg)', 'Spd of Max Gust (km/h)']

# Aggregate from incident-level to daily level
# Each row becomes one day with a total incident count and one weather snapshot
# Weather values are the same for all incidents on the same day so we just take the first
daily = df.groupby('date').agg(
    incident_count=('Count', 'sum'),
    day_of_week=('day_of_week', 'first'),
    season=('season', 'first'),
    Year=('Year', 'first'),
    Month=('Month', 'first'),
    Day=('Day', 'first'),
    **{col: (col, 'first') for col in weather_cols}
).reset_index()

print(daily.shape)
daily.head()

(3174, 16)


,date,incident_count,day_of_week,season,Year,Month,Day,Max Temp (°C),Min Temp (°C),Mean Temp (°C),Total Rain (mm),Total Snow (cm),Total Precip (mm),Snow on Grnd (cm),Dir of Max Gust (10s deg),Spd of Max Gust (km/h)
0,2016-12-06,11,Tuesday,Winter,2016,12,6,-18.2,-21.4,-19.8,0.0,0.0,0.0,3,35,48
1,2016-12-07,24,Wednesday,Winter,2016,12,7,-18.6,-25.0,-21.8,0.0,0.0,0.0,3,34,33
2,2016-12-08,27,Thursday,Winter,2016,12,8,-19.8,-26.7,-23.3,0.0,0.0,0.0,3,0,0
3,2016-12-09,41,Friday,Winter,2016,12,9,-20.2,-24.4,-22.3,0.0,0.4,0.4,3,16,30
4,2016-12-10,40,Saturday,Winter,2016,12,10,-18.3,-23.6,-21.0,0.0,0.0,0.0,3,0,0


In [ ]:
# Add a binary flag for weekends — 1 if Saturday or Sunday, 0 otherwise
daily['is_weekend'] = daily['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# Convert day_of_week and season from text to numeric 0/1 columns
# Models can't do math with words like "Tuesday" or "Winter"
# drop_first=True removes one category from each group since it's redundant
daily = pd.get_dummies(daily, columns=['day_of_week', 'season'], prefix=['dow', 'season'], drop_first=True)

print(daily.columns.tolist())

['date', 'incident_count', 'Year', 'Month', 'Day', 'Max Temp (°C)', 'Min Temp (°C)', 'Mean Temp (°C)', 'Total Rain (mm)', 'Total Snow (cm)', 'Total Precip (mm)', 'Snow on Grnd (cm)', 'Dir of Max Gust (10s deg)', 'Spd of Max Gust (km/h)', 'is_weekend', 'dow_Monday', 'dow_Saturday', 'dow_Sunday', 'dow_Thursday', 'dow_Tuesday', 'dow_Wednesday', 'season_Spring', 'season_Summer', 'season_Winter']


In [ ]:
from sklearn.model_selection import train_test_split

# X is all input features — everything the model learns from
# We drop date and Year (not useful patterns) and incident_count (that's what we're predicting)
X = daily.drop(columns=['date', 'Year', 'incident_count'])

# y is the target — the number of incidents we're trying to predict
y = daily['incident_count']

# Split 80% into training (model learns from this) and 20% into test (model is evaluated on this)
# random_state=42 fixes the random split so results are reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training rows: {X_train.shape[0]}')
print(f'Test rows: {X_test.shape[0]}')

Training rows: 2539
Test rows: 635


In [ ]:
lr = LinearRegression()

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

sgd = SGDRegressor(
    loss="epsilon_insensitive",
    penalty="l1",
    random_state=42
)

In [ ]:
from sklearn.ensemble import VotingRegressor

# Voting Regressor - uses different models and gives the average predicted values
# Each estimator is a regressor previously used in the notebook
vote = VotingRegressor(estimators=[('lr', lr), ('rf', rf), ('sgd', sgd)])

# Trains the model on the same data as the other models, and predicts based on the same test set
vote.fit(X_train, y_train)
y_pred_vote = vote.predict(X_test)

# Provides the same metrics as other models
mae  = mean_absolute_error(y_test, y_pred_vote)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_vote))
r2   = r2_score(y_test, y_pred_vote)

print('=== Voting Regressor ===')
print(f'Mean Absolute Error:  {mae:.2f}')
print(f'Root Mean Squared Error:  {rmse:.2f}')
print(f'R2 Score:  {r2:.3f}')

=== Voting Regressor ===
Mean Absolute Error:  5.16
Root Mean Squared Error:  6.88
R2 Score:  0.403


In [ ]:
def predict_new(data_dict):
    df = pd.DataFrame([data_dict])
    df = df.reindex(columns=X_train.columns, fill_value=0)
    return vote.predict(df)[0]

In [ ]:
import pandas as pd

new_data = pd.DataFrame([{
    'Month': 2,
    'Day': 15,

    'Max Temp (°C)': -5,
    'Min Temp (°C)': -15,
    'Mean Temp (°C)': -10,
    'Total Rain (mm)': 0,
    'Total Snow (cm)': 3,
    'Total Precip (mm)': 3,
    'Snow on Grnd (cm)': 10,
    'Dir of Max Gust (10s deg)': 20,
    'Spd of Max Gust (km/h)': 25,

    'is_weekend': 0,
    'dow_Monday': 0,
    'dow_Tuesday': 0,
    'dow_Wednesday': 0,
    'dow_Thursday': 0,
    'dow_Friday': 1,

    'season_Spring': 0,
    'season_Summer': 0,
    'season_Winter': 1
}])

# Match training columns exactly
new_data = new_data.reindex(columns=X_train.columns, fill_value=0)

prediction = vote.predict(new_data)
print("Winter weekday prediction:", prediction[0])

Winter weekday prediction: 23.680409959153355


In [ ]:
new_data = pd.DataFrame([{
    'Month': 1,
    'Day': 10,

    'Max Temp (°C)': -15,
    'Min Temp (°C)': -25,
    'Mean Temp (°C)': -20,
    'Total Rain (mm)': 0,
    'Total Snow (cm)': 8,
    'Total Precip (mm)': 8,
    'Snow on Grnd (cm)': 20,
    'Dir of Max Gust (10s deg)': 30,
    'Spd of Max Gust (km/h)': 40,

    'is_weekend': 1,

    'season_Winter': 1
}])

new_data = new_data.reindex(columns=X_train.columns, fill_value=0)
print("Snowy weekend prediction:", vote.predict(new_data)[0])

Snowy weekend prediction: 33.346501562989864


In [ ]:
new_data = pd.DataFrame([{
    'Month': 7,
    'Day': 10,

    'Max Temp (°C)': 25,
    'Min Temp (°C)': 15,
    'Mean Temp (°C)': 20,
    'Total Rain (mm)': 0,
    'Total Snow (cm)': 0,
    'Total Precip (mm)': 0,
    'Snow on Grnd (cm)': 0,
    'Dir of Max Gust (10s deg)': 10,
    'Spd of Max Gust (km/h)': 10,

    'is_weekend': 0,

    'season_Summer': 1
}])

new_data = new_data.reindex(columns=X_train.columns, fill_value=0)
print("Summer day prediction:", vote.predict(new_data)[0])

Summer day prediction: 19.38099988819344
